# Base Agents Training & Evaluation - ALL FOLDS
## PPO vs Baseline Strategies

**Date:** November 10, 2025  
**Objective:** Train PPO agent on all 50 folds, implement baselines, compare performance

---

## Contents
1. Environment Setup & Configuration
2. PPO Agent Training on All Folds
3. Baseline Strategies Implementation
4. Performance Evaluation (Select Fold)
5. Comparative Analysis
6. Portfolio Weights Visualization
7. Risk-Return Analysis

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# RL imports
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
import torch

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Project imports
from harlf.config import TICKERS, get_fold_files
from harlf.envs.portfolio_env import PortfolioEnv
from harlf.agents.dirichlet_policy import SoftmaxActorCriticPolicy

print("✅ Imports complete")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 1. Configuration

In [ ]:
# ============================================================================
# TRAINING CONFIGURATION
# ============================================================================

# Training settings
TRAIN_ALL_FOLDS = True  # Set to True to train all 50 folds, False for single fold
N_FOLDS = 50  # Total number of folds
START_FOLD = 0  # Starting fold (useful if resuming)
END_FOLD = 50  # Ending fold (exclusive)

# Single fold settings (used if TRAIN_ALL_FOLDS=False)
SINGLE_FOLD_ID = 0

# Training hyperparameters
REWARD_TYPE = 'ema_sharpe'
TOTAL_TIMESTEPS = 200000
SEED = 42

# Model save directory
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Set seeds
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"📊 Configuration")
print("="*70)
print(f"Train all folds: {TRAIN_ALL_FOLDS}")
if TRAIN_ALL_FOLDS:
    print(f"Fold range: {START_FOLD} to {END_FOLD-1}")
    print(f"Total folds to train: {END_FOLD - START_FOLD}")
else:
    print(f"Single fold: {SINGLE_FOLD_ID}")
print(f"Reward: {REWARD_TYPE}")
print(f"Training timesteps per fold: {TOTAL_TIMESTEPS:,}")
print(f"Seed: {SEED}")
print(f"Models directory: {MODELS_DIR.absolute()}")

## 2. Environment Creation Function

In [ ]:
# Create environments
def make_env(fold_id, split, reward_type='ema_sharpe'):
    """Create a portfolio environment for a given fold and split."""
    env = PortfolioEnv(fold_id=fold_id, split=split, reward_type=reward_type)
    env = Monitor(env)
    return env

print("✅ Environment creation function defined")

## 3. PPO Training on All Folds

This cell will train a PPO agent on each fold from START_FOLD to END_FOLD.

In [ ]:
if TRAIN_ALL_FOLDS:
    print("\n" + "="*70)
    print("🏋️  TRAINING PPO ON MULTIPLE FOLDS")
    print("="*70)
    
    # PPO hyperparameters (defined once, used for all folds)
    policy_kwargs = {
        'net_arch': [256, 256],
        'activation_fn': torch.nn.Tanh,
        'ortho_init': True,
    }
    
    ppo_config = {
        'policy': SoftmaxActorCriticPolicy,
        'learning_rate': 3e-4,
        'n_steps': 2048,
        'batch_size': 128,
        'n_epochs': 10,
        'gamma': 0.985,
        'gae_lambda': 0.95,
        'clip_range': 0.2,
        'ent_coef': 0.01,
        'vf_coef': 0.5,
        'max_grad_norm': 0.5,
        'policy_kwargs': policy_kwargs,
        'verbose': 1,
        'seed': SEED,
    }
    
    # Track training summary
    training_summary = []
    
    # Loop through all folds
    for fold_id in range(START_FOLD, END_FOLD):
        print(f"\n{'='*70}")
        print(f"📊 FOLD {fold_id}/{N_FOLDS-1} ({fold_id - START_FOLD + 1}/{END_FOLD - START_FOLD})")
        print(f"{'='*70}")
        
        # Check if model already exists
        fold_model_dir = MODELS_DIR / f'fold_{fold_id}'
        fold_model_dir.mkdir(parents=True, exist_ok=True)
        model_path = fold_model_dir / f'ppo_fold_{fold_id}_final.zip'
        
        if model_path.exists():
            print(f"⏩ Model already exists at {model_path}")
            print(f"   Skipping training for fold {fold_id}")
            training_summary.append({
                'fold': fold_id,
                'status': 'skipped',
                'path': str(model_path)
            })
            continue
        
        try:
            # Create training environment for this fold
            print(f"\n🔧 Creating training environment for fold {fold_id}...")
            train_env = DummyVecEnv([lambda fold=fold_id: make_env(fold, 'train', REWARD_TYPE)])
            print(f"   Observation space: {train_env.observation_space.shape}")
            print(f"   Action space: {train_env.action_space.shape}")
            
            # Create PPO model
            print(f"\n🤖 Initializing PPO model...")
            model = PPO(env=train_env, **ppo_config)
            
            # Train
            print(f"\n🏋️  Training for {TOTAL_TIMESTEPS:,} timesteps...")
            print(f"   This may take 10-20 minutes per fold...\n")
            model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)
            
            # Save model
            print(f"\n💾 Saving model to {model_path}...")
            model.save(str(model_path))
            
            print(f"\n✅ Fold {fold_id} training complete!")
            training_summary.append({
                'fold': fold_id,
                'status': 'success',
                'path': str(model_path)
            })
            
        except Exception as e:
            print(f"\n❌ Error training fold {fold_id}: {e}")
            training_summary.append({
                'fold': fold_id,
                'status': 'failed',
                'error': str(e)
            })
            
        finally:
            # Clean up to free memory
            if 'train_env' in locals():
                train_env.close()
                del train_env
            if 'model' in locals():
                del model
            
            # Force garbage collection
            import gc
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    # Print training summary
    print(f"\n{'='*70}")
    print("🎉 TRAINING COMPLETE - SUMMARY")
    print(f"{'='*70}\n")
    
    summary_df = pd.DataFrame(training_summary)
    print(summary_df.to_string(index=False))
    
    # Count successes/failures
    n_success = sum(1 for s in training_summary if s['status'] == 'success')
    n_skipped = sum(1 for s in training_summary if s['status'] == 'skipped')
    n_failed = sum(1 for s in training_summary if s['status'] == 'failed')
    
    print(f"\n📊 Results:")
    print(f"   ✅ Successfully trained: {n_success}")
    print(f"   ⏩ Skipped (already exist): {n_skipped}")
    print(f"   ❌ Failed: {n_failed}")
    print(f"   📁 Models saved in: {MODELS_DIR.absolute()}")
    
else:
    print("⏩ Skipping multi-fold training (TRAIN_ALL_FOLDS=False)")
    print(f"   Set TRAIN_ALL_FOLDS=True to train on all folds")

## 4. Train or Load Single Fold Model

For evaluation and analysis, we'll work with a specific fold.

In [ ]:
# ============================================================================
# EVALUATION CONFIGURATION
# ============================================================================

# Select which fold to evaluate
EVAL_FOLD_ID = 0  # Change this to evaluate different folds

print(f"\n{'='*70}")
print(f"📊 EVALUATION SETUP - Fold {EVAL_FOLD_ID}")
print(f"{'='*70}\n")

# Create environments for evaluation
train_env = DummyVecEnv([lambda: make_env(EVAL_FOLD_ID, 'train', REWARD_TYPE)])
val_env = make_env(EVAL_FOLD_ID, 'val', REWARD_TYPE)
test_env = make_env(EVAL_FOLD_ID, 'test', REWARD_TYPE)

print(f"✅ Environments created for fold {EVAL_FOLD_ID}")
print(f"   Train env: {train_env.observation_space.shape}")
print(f"   Val env: {val_env.observation_space.shape}")
print(f"   Test env: {test_env.observation_space.shape}")

# Load the trained model
model_path = MODELS_DIR / f'fold_{EVAL_FOLD_ID}' / f'ppo_fold_{EVAL_FOLD_ID}_final.zip'

if model_path.exists():
    print(f"\n📦 Loading model from {model_path}...")
    model = PPO.load(str(model_path))
    print("✅ Model loaded successfully")
else:
    print(f"\n❌ Model not found at {model_path}")
    print(f"   Please train fold {EVAL_FOLD_ID} first or check the path")
    raise FileNotFoundError(f"Model not found: {model_path}")

## 5. Baseline Strategies

Implement simple baseline strategies for comparison.

In [ ]:
class EqualWeightStrategy:
    """Equal weight (1/N) strategy."""
    
    def __init__(self, n_assets=7):
        self.n_assets = n_assets
        self.weights = np.ones(n_assets) / n_assets
    
    def predict(self, observation, deterministic=True):
        """Predict action (deterministic parameter ignored for compatibility)."""
        return self.weights, None


class RandomStrategy:
    """Random portfolio weights."""
    
    def __init__(self, n_assets=7, seed=42):
        self.n_assets = n_assets
        self.rng = np.random.RandomState(seed)
    
    def predict(self, observation, deterministic=True):
        """Predict action (deterministic parameter ignored for compatibility)."""
        weights = self.rng.dirichlet(np.ones(self.n_assets))
        return weights, None


class MomentumStrategy:
    """Simple momentum: allocate to assets with positive 21-day returns."""
    
    def __init__(self, n_assets=7):
        self.n_assets = n_assets
    
    def predict(self, observation, deterministic=True):
        """Predict action (deterministic parameter ignored for compatibility)."""
        # Extract return_21d for each asset (feature index 1)
        # Observation is flattened: [asset1_feats, asset2_feats, ...]
        n_features = len(observation) // self.n_assets
        returns = [observation[i * n_features + 1] for i in range(self.n_assets)]
        
        # Allocate to positive momentum assets
        positive_returns = np.array([max(r, 0) for r in returns])
        
        if positive_returns.sum() > 0:
            weights = positive_returns / positive_returns.sum()
        else:
            weights = np.ones(self.n_assets) / self.n_assets
        
        return weights, None


print("✅ Baseline strategies defined")
print("   - Equal Weight (1/N)")
print("   - Random")
print("   - Momentum (21-day)")

## 6. Performance Evaluation

Evaluate all agents on validation and test sets.

In [ ]:
def evaluate_agent(agent, env, agent_name, n_episodes=1, deterministic=True):
    """
    Evaluate an agent on an environment.
    
    Returns:
        Dictionary with performance metrics and episode data
    """
    all_metrics = []
    all_returns = []
    all_weights = []
    all_values = []
    
    for episode in range(n_episodes):
        obs, info = env.reset()
        done = False
        truncated = False
        
        episode_returns = []
        episode_weights = []
        episode_values = [info['portfolio_value']]
        
        while not (done or truncated):
            action, _states = agent.predict(obs, deterministic=deterministic)
            obs, reward, done, truncated, info = env.step(action)
            
            episode_returns.append(info['portfolio_return'])
            episode_weights.append(info['weights'])
            episode_values.append(info['portfolio_value'])
        
        # Get episode metrics from unwrapped env
        unwrapped_env = env.unwrapped if hasattr(env, 'unwrapped') else env
        if hasattr(unwrapped_env, 'get_episode_metrics'):
            metrics = unwrapped_env.get_episode_metrics()
            all_metrics.append(metrics)
        
        all_returns.append(episode_returns)
        all_weights.append(episode_weights)
        all_values.append(episode_values)
    
    # Average metrics across episodes
    if all_metrics:
        avg_metrics = {k: np.mean([m[k] for m in all_metrics]) for k in all_metrics[0].keys()}
    else:
        avg_metrics = {}
    
    return {
        'agent_name': agent_name,
        'metrics': avg_metrics,
        'returns': all_returns[0] if n_episodes == 1 else all_returns,
        'weights': all_weights[0] if n_episodes == 1 else all_weights,
        'values': all_values[0] if n_episodes == 1 else all_values,
    }

print("✅ Evaluation function defined")

In [ ]:
# Evaluate all agents on validation set
print(f"\n🔍 Evaluating agents on VALIDATION set (Fold {EVAL_FOLD_ID})...\n")

agents = {
    'PPO': model,
    'Equal Weight': EqualWeightStrategy(),
    'Random': RandomStrategy(seed=SEED),
    'Momentum': MomentumStrategy(),
}

val_results = {}
for name, agent in agents.items():
    print(f"Evaluating {name}...")
    result = evaluate_agent(agent, val_env, name, n_episodes=1, deterministic=True)
    val_results[name] = result
    
    metrics = result['metrics']
    print(f"  Sharpe: {metrics.get('sharpe_ratio', 0):.4f}")
    print(f"  Return: {metrics.get('total_return', 0):.4f}")
    print(f"  Drawdown: {metrics.get('max_drawdown', 0):.4f}")
    print()

print("✅ Validation evaluation complete")

In [ ]:
# Create comparison table
comparison_data = []
for name, result in val_results.items():
    metrics = result['metrics']
    comparison_data.append({
        'Agent': name,
        'Sharpe Ratio': metrics.get('sharpe_ratio', 0),
        'Total Return': metrics.get('total_return', 0),
        'Ann. Return': metrics.get('total_return', 0) * (252 / metrics.get('episode_length', 1)),
        'Volatility': metrics.get('volatility', 0),
        'Max Drawdown': metrics.get('max_drawdown', 0),
        'Sortino Ratio': metrics.get('sortino_ratio', 0),
        'Avg Turnover': metrics.get('mean_turnover', 0),
        'Episode Length': metrics.get('episode_length', 0),
    })

comparison_df = pd.DataFrame(comparison_data).set_index('Agent')
comparison_df = comparison_df.sort_values('Sharpe Ratio', ascending=False)

print(f"\n📊 VALIDATION SET PERFORMANCE COMPARISON (Fold {EVAL_FOLD_ID})")
print("="*100)
display(comparison_df.style.format({
    'Sharpe Ratio': '{:.4f}',
    'Total Return': '{:.4f}',
    'Ann. Return': '{:.4f}',
    'Volatility': '{:.4f}',
    'Max Drawdown': '{:.4f}',
    'Sortino Ratio': '{:.4f}',
    'Avg Turnover': '{:.4f}',
    'Episode Length': '{:.0f}',
}))

## 7. Comparative Visualization

In [ ]:
# Plot 1: Sharpe Ratio comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sharpe Ratio
comparison_df['Sharpe Ratio'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Sharpe Ratio Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Sharpe Ratio')
axes[0].set_xlabel('')
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Total Return
comparison_df['Total Return'].plot(kind='bar', ax=axes[1], color='green', edgecolor='black')
axes[1].set_title('Total Return Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Total Return')
axes[1].set_xlabel('')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

# Max Drawdown
comparison_df['Max Drawdown'].plot(kind='bar', ax=axes[2], color='red', edgecolor='black')
axes[2].set_title('Max Drawdown Comparison', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Max Drawdown')
axes[2].set_xlabel('')
axes[2].axhline(y=-0.2, color='orange', linestyle='--', alpha=0.5, label='Threshold')
axes[2].grid(axis='y', alpha=0.3)
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend()

plt.suptitle(f'Validation Set Performance Metrics (Fold {EVAL_FOLD_ID})', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Cumulative returns
fig, ax = plt.subplots(figsize=(16, 6))

for name, result in val_results.items():
    values = result['values']
    returns = [(v / 1_000_000 - 1) * 100 for v in values]  # Convert to % return from $1M
    ax.plot(returns, label=name, linewidth=2, alpha=0.8)

ax.set_title(f'Cumulative Returns - Validation Set (Fold {EVAL_FOLD_ID})', fontsize=14, fontweight='bold')
ax.set_xlabel('Days', fontsize=12)
ax.set_ylabel('Return from Initial (%)', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(alpha=0.3)
ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Risk-Return scatter
fig, ax = plt.subplots(figsize=(10, 8))

for name in comparison_df.index:
    volatility = comparison_df.loc[name, 'Volatility']
    ann_return = comparison_df.loc[name, 'Ann. Return']
    sharpe = comparison_df.loc[name, 'Sharpe Ratio']
    
    ax.scatter(volatility, ann_return, s=200, alpha=0.7, label=name)
    ax.annotate(f"{name}\n(SR={sharpe:.2f})", 
                (volatility, ann_return), 
                textcoords="offset points", 
                xytext=(0,10), 
                ha='center',
                fontsize=9)

ax.set_title(f'Risk-Return Profile - Validation Set (Fold {EVAL_FOLD_ID})', fontsize=14, fontweight='bold')
ax.set_xlabel('Annualized Volatility', fontsize=12)
ax.set_ylabel('Annualized Return', fontsize=12)
ax.grid(alpha=0.3)
ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax.axvline(x=0, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Portfolio Weights Analysis

Analyze how different agents allocate across assets.

In [ ]:
# Plot portfolio weights over time for each agent
n_agents = len(val_results)
fig, axes = plt.subplots(n_agents, 1, figsize=(16, 4*n_agents))

if n_agents == 1:
    axes = [axes]

for i, (name, result) in enumerate(val_results.items()):
    weights = np.array(result['weights'])
    
    # Create stacked area plot
    axes[i].stackplot(range(len(weights)), weights.T, labels=TICKERS, alpha=0.7)
    axes[i].set_title(f'{name} - Portfolio Allocation Over Time', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Weight')
    axes[i].set_ylim(0, 1)
    axes[i].legend(loc='upper left', ncol=7, fontsize=9)
    axes[i].grid(alpha=0.3)

axes[-1].set_xlabel('Days')

plt.suptitle(f'Portfolio Weights Evolution - Validation Set (Fold {EVAL_FOLD_ID})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Average weights across episode
avg_weights_data = []

for name, result in val_results.items():
    weights = np.array(result['weights'])
    avg_weights = weights.mean(axis=0)
    
    for i, ticker in enumerate(TICKERS):
        avg_weights_data.append({
            'Agent': name,
            'Ticker': ticker,
            'Avg Weight': avg_weights[i]
        })

avg_weights_df = pd.DataFrame(avg_weights_data)
avg_weights_pivot = avg_weights_df.pivot(index='Agent', columns='Ticker', values='Avg Weight')

print(f"\n📊 AVERAGE PORTFOLIO WEIGHTS - Validation (Fold {EVAL_FOLD_ID})")
print("="*80)
display(avg_weights_pivot.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=1))

In [ ]:
# Visualize average weights
fig, ax = plt.subplots(figsize=(12, 6))

avg_weights_pivot.T.plot(kind='bar', ax=ax, width=0.8, edgecolor='black')
ax.set_title('Average Portfolio Weights by Agent', fontsize=14, fontweight='bold')
ax.set_xlabel('Ticker', fontsize=12)
ax.set_ylabel('Average Weight', fontsize=12)
ax.legend(title='Agent', fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 9. Test Set Evaluation

Final evaluation on held-out test set.

In [ ]:
# Evaluate on test set
print(f"\n🔍 Evaluating agents on TEST set (Fold {EVAL_FOLD_ID})...\n")

test_results = {}
for name, agent in agents.items():
    print(f"Evaluating {name}...")
    result = evaluate_agent(agent, test_env, name, n_episodes=1, deterministic=True)
    test_results[name] = result
    
    metrics = result['metrics']
    print(f"  Sharpe: {metrics.get('sharpe_ratio', 0):.4f}")
    print(f"  Return: {metrics.get('total_return', 0):.4f}")
    print(f"  Drawdown: {metrics.get('max_drawdown', 0):.4f}")
    print()

print("✅ Test evaluation complete")

In [ ]:
# Create test set comparison table
test_comparison_data = []
for name, result in test_results.items():
    metrics = result['metrics']
    test_comparison_data.append({
        'Agent': name,
        'Sharpe Ratio': metrics.get('sharpe_ratio', 0),
        'Total Return': metrics.get('total_return', 0),
        'Ann. Return': metrics.get('total_return', 0) * (252 / metrics.get('episode_length', 1)),
        'Volatility': metrics.get('volatility', 0),
        'Max Drawdown': metrics.get('max_drawdown', 0),
        'Sortino Ratio': metrics.get('sortino_ratio', 0),
        'Avg Turnover': metrics.get('mean_turnover', 0),
        'Episode Length': metrics.get('episode_length', 0),
    })

test_comparison_df = pd.DataFrame(test_comparison_data).set_index('Agent')
test_comparison_df = test_comparison_df.sort_values('Sharpe Ratio', ascending=False)

print(f"\n📊 TEST SET PERFORMANCE COMPARISON (Fold {EVAL_FOLD_ID})")
print("="*100)
display(test_comparison_df.style.format({
    'Sharpe Ratio': '{:.4f}',
    'Total Return': '{:.4f}',
    'Ann. Return': '{:.4f}',
    'Volatility': '{:.4f}',
    'Max Drawdown': '{:.4f}',
    'Sortino Ratio': '{:.4f}',
    'Avg Turnover': '{:.4f}',
    'Episode Length': '{:.0f}',
}))

In [ ]:
# Plot test set cumulative returns
fig, ax = plt.subplots(figsize=(16, 6))

for name, result in test_results.items():
    values = result['values']
    returns = [(v / 1_000_000 - 1) * 100 for v in values]
    ax.plot(returns, label=name, linewidth=2, alpha=0.8)

ax.set_title(f'Cumulative Returns - Test Set (Fold {EVAL_FOLD_ID}, 21 Days)', fontsize=14, fontweight='bold')
ax.set_xlabel('Days', fontsize=12)
ax.set_ylabel('Return from Initial (%)', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(alpha=0.3)
ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 10. Summary

### Key Notes

1. **Multi-Fold Training**: This notebook trains PPO agents on all 50 folds
2. **Model Storage**: Models are saved to `../models/fold_X/ppo_fold_X_final.zip`
3. **Evaluation**: Change `EVAL_FOLD_ID` to evaluate different folds
4. **Resume Training**: If training is interrupted, set `START_FOLD` to resume from a specific fold

### Next Steps for Full Walk-Forward Analysis

1. After training all folds, create a walk-forward aggregation notebook
2. Load and evaluate all 50 trained models
3. Compute aggregate statistics across all folds
4. Analyze performance consistency and robustness

---

**Notebook Complete** ✅